In [6]:
import os
import shutil
import pandas as pd
from pathlib import Path

def smart_merge_and_rename(src_5digit_img, csv_5digit_path, 
                           src_6digit_img, csv_6digit_path, 
                           output_dir, mode="train"):
    
    out_path = Path(output_dir)
    out_img_path = out_path / "images"
    out_img_path.mkdir(parents=True, exist_ok=True)
    
    datasets = [
        (Path(src_5digit_img), csv_5digit_path, "5d"),
        (Path(src_6digit_img), csv_6digit_path, "6d")
    ]
    
    new_records = []
    
    print(f"--- Đang xử lý tập [{mode.upper()}] ---")
    
    for img_dir, csv_path, suffix in datasets:
        # Đọc file CSV mà không ép header, đọc toàn bộ dưới dạng string để tránh lỗi số học
        df = pd.read_csv(csv_path, header=None, dtype=str)
        
        # In ra 2 dòng đầu tiên của file CSV để kiểm tra cấu trúc cột thực tế
        print(f"Đọc file {Path(csv_path).name}, 2 dòng đầu:")
        print(df.head(2))
        
        # Tự động tìm cột nào chứa tên file ảnh (cột có chứa đuôi .jpg hoặc tên file khớp với thư mục)
        filename_col = None
        label_col = None
        
        for col in df.columns:
            # Kiểm tra xem cột này có chứa các giá trị giống tên file trong thư mục không
            sample_val = str(df[col].iloc[0])
            if ".jpg" in sample_val or ".png" in sample_val or any((img_dir / sample_val).exists() for sample_val in df[col].dropna().head(5)):
                filename_col = col
                break
                
        # Nếu không tìm thấy bằng đuôi mở rộng, mặc định lấy cột 0 là tên file, cột 1 là nhãn
        if filename_col is None:
            filename_col = 0
            label_col = 1
        else:
            # Chọn cột còn lại làm nhãn (ưu tiên cột cạnh bên hoặc cột chứa giá trị khác tên file)
            candidates = [c for c in df.columns if c != filename_col]
            label_col = candidates[0] if candidates else 1
            
        print(f"-> Tự nhận diện: Cột tên file = {filename_col}, Cột nhãn = {label_col}\n")
        
        # Tạo dictionary tra cứu: {tên_file_gốc: nhãn_thực_sự}
        label_dict = dict(zip(df[filename_col].str.strip(), df[label_col].str.strip()))
        
        idx = 0
        for img_path in img_dir.glob("*.*"):
            old_name = img_path.name
            
            if old_name not in label_dict:
                continue
                
            new_file_name = f"{mode}_{idx}_{suffix}.jpg"
            destination_path = out_img_path / new_file_name
            
            # Copy ảnh sang thư mục gộp với tên mới
            shutil.copy(img_path, destination_path)
            
            new_records.append({
                'filename': new_file_name,
                'label': label_dict[old_name]
            })
            
            idx += 1
            
    # Xuất file CSV kết quả
    final_df = pd.DataFrame(new_records)
    final_csv_path = out_path / f"{mode}_labels.csv"
    final_df.to_csv(final_csv_path, index=False)
    
    print(f"Đã hoàn tất! Lưu tại: {out_path} (Tổng số lượng: {len(new_records)} mẫu)\n")

# Chạy lại cho tập Train
smart_merge_and_rename(
    src_5digit_img="/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_5-digit/train/5-digit_train_img",
    csv_5digit_path="/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_5-digit/train/5-digit_train_rec_label_CSV.csv",
    src_6digit_img="/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_6-digit/train/6-digit_train_img",
    csv_6digit_path="/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_6-digit/train/6-digit_train_rec_label_CSV.csv",
    output_dir="/Users/mac/MeterReadAI/data/DataForTrain/recognition_merged_train",
    mode="train"
)

# Chạy lại cho tập Test
smart_merge_and_rename(
    src_5digit_img="/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_5-digit/test/5-digit_test_img",
    csv_5digit_path="/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_5-digit/test/5-digit_test_rec_label_CSV.csv",
    src_6digit_img="/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_6-digit/test/6-digit_test_img",
    csv_6digit_path="/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_6-digit/test/6-digit_test_rec_label_CSV.csv",
    output_dir="/Users/mac/MeterReadAI/data/DataForTrain/recognition_merged_test",
    mode="test"
)

--- Đang xử lý tập [TRAIN] ---
Đọc file 5-digit_train_rec_label_CSV.csv, 2 dòng đầu:
            0      1  2  3  4
0  train0.jpg  00000  0  0  0
1  train1.jpg  00000  0  0  0
-> Tự nhận diện: Cột tên file = 0, Cột nhãn = 1

Đọc file 6-digit_train_rec_label_CSV.csv, 2 dòng đầu:
            0       1  2  3  4
0  train0.jpg  002340  1  1  1
1  train1.jpg  000001  1  1  1
-> Tự nhận diện: Cột tên file = 0, Cột nhãn = 1

Đã hoàn tất! Lưu tại: /Users/mac/MeterReadAI/data/DataForTrain/recognition_merged_train (Tổng số lượng: 14387 mẫu)

--- Đang xử lý tập [TEST] ---
Đọc file 5-digit_test_rec_label_CSV.csv, 2 dòng đầu:
           0      1  2  3  4
0  test0.jpg  00000  0  0  0
1  test1.jpg  00000  0  0  0
-> Tự nhận diện: Cột tên file = 0, Cột nhãn = 1

Đọc file 6-digit_test_rec_label_CSV.csv, 2 dòng đầu:
           0       1  2  3  4
0  test0.jpg  203976  1  0  1
1  test1.jpg  324715  1  0  1
-> Tự nhận diện: Cột tên file = 0, Cột nhãn = 1

Đã hoàn tất! Lưu tại: /Users/mac/MeterReadAI/data/Dat